# LeetCode 2171 — Removing Minimum Number of Magic Beans bằng Python

Link đề: https://leetcode.com/problems/removing-minimum-number-of-magic-beans/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting, Enumeration, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minimumRemoval(self, beans: List[int]) -> int:
        ...
```

Chọn một mức `t`: mọi túi có `< t` hạt phải **đổ hết**, mọi túi có `≥ t` hạt bớt về đúng `t`. Bài toán thu về: **tối đa số hạt giữ lại** `t × (số túi ≥ t)`. Notebook đi từ **thử mọi `t` từ 1 đến max** → **chỉ thử các giá trị có trong mảng** → **sắp xếp + công thức `beans[i] × (n − i)`** → **đếm theo giá trị (counting) `O(n + A)`** → **tiền tố + tìm nhị phân** (phiên bản truy vấn).

## 1. Mục tiêu học

1. Đổi bài "bỏ ít nhất" thành "**giữ nhiều nhất**": `đáp án = tổng − max giữ lại`.
2. Chứng minh mức `t` tối ưu luôn là **một giá trị có sẵn** trong mảng (không cần thử mọi số).
3. Sau khi sắp tăng, chọn `t = beans[i]` thì số túi giữ là `n − i` ⇒ giữ `beans[i] × (n − i)`.
4. Cài đặt bản `O(n + A)` bằng đếm theo giá trị khi `A = max ≤ 10⁵`.
5. Nhận diện các lỗi: chọn trung bình/trung vị, chỉ bớt túi lớn mà quên đổ túi nhỏ, nhầm công thức số túi.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `beans`, `beans[i]` là số hạt đậu trong túi `i`. Bỏ bớt hạt (có thể không bỏ) ở mỗi túi sao cho **mọi túi còn hạt** (không rỗng) có **cùng số hạt**. Hạt đã bỏ không được trả lại. Trả về **số hạt ít nhất** phải bỏ.

**Ví dụ 1:** `beans = [4,1,6,5]` → `4` (đổ túi 1 hạt, bớt 6 → 4, bớt 5 → 4: `1 + 2 + 1`).

**Ví dụ 2:** `beans = [2,10,3,2]` → `7` (đổ hai túi 2 hạt và túi 3 hạt: `2 + 2 + 3`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ beans[i] ≤ 10⁵`. Tổng có thể tới `10¹⁰` (cần `long` ở C++/Java).

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

examples = [([4, 1, 6, 5], 4), ([2, 10, 3, 2], 7)]

def removal_for(beans, t):
    """Số hạt bỏ nếu mọi túi còn hạt có đúng t hạt (t = 0: đổ hết)."""
    return sum(b if b < t or t == 0 else b - t for b in beans)

for beans, exp in examples:
    print(f"{beans}: tổng={sum(beans)}, " + ", ".join(f"t={t}->{removal_for(beans, t)}" for t in sorted(set(beans))), f"| kỳ vọng {exp}")

[4, 1, 6, 5]: tổng=16, t=1->12, t=4->4, t=5->6, t=6->10 | kỳ vọng 4
[2, 10, 3, 2]: tổng=17, t=2->9, t=3->11, t=10->7 | kỳ vọng 7


## 3. Kiến thức nền

### 3.1 Sắp xếp và chỉ số "số phần tử ≥ x"

Sau khi sắp tăng, mọi phần tử từ vị trí `i` trở đi đều `≥ beans[i]` ⇒ có `n − i` phần tử như vậy (nếu `i` là vị trí **đầu tiên** của giá trị đó thì là "số phần tử `≥` giá trị").

In [2]:
a = sorted([2, 10, 3, 2])
for i, v in enumerate(a):
    print(f"  i={i}, beans[i]={v}: n - i = {len(a) - i} túi có ≥ {v} hạt (tính từ vị trí i)")

  i=0, beans[i]=2: n - i = 4 túi có ≥ 2 hạt (tính từ vị trí i)
  i=1, beans[i]=2: n - i = 3 túi có ≥ 2 hạt (tính từ vị trí i)
  i=2, beans[i]=3: n - i = 2 túi có ≥ 3 hạt (tính từ vị trí i)
  i=3, beans[i]=10: n - i = 1 túi có ≥ 10 hạt (tính từ vị trí i)


### 3.2 `bisect_left` — đếm số phần tử nhỏ hơn `t`

In [3]:
a = [1, 4, 5, 6]
for t in [0, 1, 4, 7]:
    k = bisect_left(a, t)
    print(f"t={t}: {k} túi < t, {len(a) - k} túi ≥ t")

t=0: 0 túi < t, 4 túi ≥ t
t=1: 0 túi < t, 4 túi ≥ t
t=4: 1 túi < t, 3 túi ≥ t
t=7: 4 túi < t, 0 túi ≥ t


### 3.3 Đếm theo giá trị (counting) và tổng hậu tố

`cnt[v]` = số túi có đúng `v` hạt; `ge[v] = cnt[v] + cnt[v+1] + …` = số túi `≥ v`. Duyệt `v` từ lớn xuống nhỏ để cộng dồn.

In [4]:
beans = [4, 1, 6, 5]
A = max(beans); cnt = [0] * (A + 2)
for b in beans:
    cnt[b] += 1
ge = [0] * (A + 2)
for v in range(A, 0, -1):
    ge[v] = ge[v + 1] + cnt[v]
print("cnt:", cnt[1:A + 1], "| ge:", ge[1:A + 1])

cnt: [1, 0, 0, 1, 1, 1] | ge: [4, 3, 3, 3, 2, 1]


## 4. Ý tưởng nền tảng / trực giác

Cố định mức `t ≥ 1`:
- túi có `b < t` không thể nâng lên ⇒ phải **đổ hết** `b` hạt;
- túi có `b ≥ t` bớt `b − t` hạt.

Số hạt **giữ lại** = `t × #{b ≥ t}`. Số hạt bỏ = `tổng − t × #{b ≥ t}`.

**Vì sao không có lựa chọn nào khác?** Với một mức `t` cố định, mỗi túi chỉ có hai số phận hợp lệ: **còn đúng `t` hạt** (nếu đủ hạt) hoặc **rỗng**. Túi có `b ≥ t` nên giữ `t` (giữ được `t` hạt luôn tốt hơn đổ hết); túi có `b < t` không thể có `t` hạt ⇒ bắt buộc rỗng. Như vậy toàn bộ bài toán chỉ còn **một biến** là `t`.

Tính tay ví dụ 1 (`tổng = 16`):

| `t` | túi `≥ t` | giữ `t × số túi` | bỏ |
|---|---|---|---|
| 1 | 4 | 4 | 12 |
| 4 | 3 (4,6,5) | 12 | **4** |
| 5 | 2 (6,5) | 10 | 6 |
| 6 | 1 | 6 | 10 |

⇒ đáp án 4 với `t = 4`. Đây là đánh đổi: `t` lớn thì mỗi túi giữ nhiều nhưng ít túi; `t` nhỏ thì nhiều túi nhưng mỗi túi ít.

In [5]:
beans = [4, 1, 6, 5]; S = sum(beans)
for t in range(1, max(beans) + 1):
    k = sum(b >= t for b in beans)
    print(f"  t={t}: {k} túi ≥ t, giữ {t * k:>2}, bỏ {S - t * k:>2}" + ("   <- tối ưu" if S - t * k == 4 else ""))

  t=1: 4 túi ≥ t, giữ  4, bỏ 12
  t=2: 3 túi ≥ t, giữ  6, bỏ 10
  t=3: 3 túi ≥ t, giữ  9, bỏ  7
  t=4: 3 túi ≥ t, giữ 12, bỏ  4   <- tối ưu
  t=5: 2 túi ≥ t, giữ 10, bỏ  6
  t=6: 1 túi ≥ t, giữ  6, bỏ 10


### 4.1 Hình dạng hàm "giữ lại" theo `t` — không đơn điệu, không "một đỉnh"

Vẽ `keep(t) = t × #{b ≥ t}` bằng thanh ngang cho một mảng tự dựng. Hàm có thể **lên xuống nhiều lần** (mỗi khi `t` vượt qua một giá trị của mảng, số túi tụt xuống đột ngột). Vì vậy không thể dùng tìm tam phân (ternary search) — phải xét **mọi** ứng viên (xem lỗi 6 ở mục 13).

In [6]:
def keep_curve(beans):
    return [(t, t * sum(b >= t for b in beans)) for t in range(1, max(beans) + 1)]

demo_curve = [2, 2, 2, 2, 5, 5, 9, 9, 9]
for t, k in keep_curve(demo_curve):
    print(f"  t={t:>2} giữ {k:>3} " + "█" * (k // 2) + ("  <- có trong mảng" if t in demo_curve else ""))

  t= 1 giữ   9 ████
  t= 2 giữ  18 █████████  <- có trong mảng
  t= 3 giữ  15 ███████
  t= 4 giữ  20 ██████████
  t= 5 giữ  25 ████████████  <- có trong mảng
  t= 6 giữ  18 █████████
  t= 7 giữ  21 ██████████
  t= 8 giữ  24 ████████████
  t= 9 giữ  27 █████████████  <- có trong mảng


## 5. Cách 1 — Brute Force / baseline: thử mọi `t` từ 1 đến `max(beans)`

In [7]:
def removal_brute(beans):
    S = sum(beans)
    best = S                                   # t = 0: đổ hết (luôn khả thi)
    for t in range(1, max(beans) + 1):
        keep = t * sum(1 for b in beans if b >= t)
        best = min(best, S - keep)
    return best

for beans_, exp in examples:
    print(removal_brute(beans_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )


**Độ phức tạp:** `O(n · A)` — `10¹⁰` trong trường hợp xấu. **Nút thắt:** (1) thử cả các `t` không có trong mảng; (2) đếm lại `#{b ≥ t}` mỗi lần.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề.** Tồn tại `t` tối ưu thuộc tập giá trị của `beans`.

*Chứng minh:* xét `t` không có trong mảng, gọi `t'` là giá trị **nhỏ nhất trong mảng** mà `≥ t` (nếu không có thì `#{b ≥ t} = 0`, giữ 0, không tốt hơn `t = max`). Khi đó `#{b ≥ t'} = #{b ≥ t}` (không có túi nào có số hạt trong `[t, t')`), còn `t' > t` ⇒ giữ được **nhiều hơn**. ∎

**Insight sắp xếp.** Sắp tăng; với `t = beans[i]` (lấy vị trí **đầu tiên** của giá trị), số túi `≥ t` là `n − i`. Với các vị trí **lặp** của cùng giá trị, `n − i` nhỏ hơn ⇒ chỉ cho giá trị giữ **nhỏ hơn hoặc bằng**, nên lấy `max` trên **mọi** `i` vẫn đúng — không cần xử lý trùng!

**Quy tắc:** đáp án `= tổng − max_i beans[i] × (n − i)` (mảng đã sắp tăng).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chỉ thử các giá trị có trong mảng (vẫn đếm lại)

In [8]:
def removal_values(beans):
    S = sum(beans)
    return min(S - t * sum(1 for b in beans if b >= t) for t in set(beans))

for beans_, exp in examples:
    print(removal_values(beans_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )


**Độ phức tạp:** `O(n · V)` với `V` = số giá trị khác nhau — vẫn `O(n²)` xấu nhất.

### 7.2 Cách 3 — Sắp xếp + công thức `beans[i] × (n − i)`

In [9]:
def removal_sort(beans):
    a = sorted(beans)
    n = len(a)
    return sum(a) - max(a[i] * (n - i) for i in range(n))

for beans_, exp in examples:
    print(removal_sort(beans_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )


**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ (bản sắp; `O(1)` nếu sắp tại chỗ).

### 7.3 Cách 4 — Đếm theo giá trị, `O(n + A)`

Duyệt `v` từ `A` xuống `1`, cộng dồn `ge` = số túi `≥ v`; giữ `v × ge`. Không cần sắp xếp.

In [10]:
def removal_counting(beans):
    A = max(beans)
    cnt = [0] * (A + 1)
    for b in beans:
        cnt[b] += 1
    best_keep = ge = 0
    for v in range(A, 0, -1):
        ge += cnt[v]
        if v * ge > best_keep:
            best_keep = v * ge
    return sum(beans) - best_keep

for beans_, exp in examples:
    print(removal_counting(beans_), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )


**Độ phức tạp:** `O(n + A)` thời gian, `O(A)` bộ nhớ. Lưu ý vòng duyệt thử **mọi** `v` (kể cả không có trong mảng) — vẫn đúng, chỉ dư vài phép.

### 7.4 Cách 5 — Tiền tố + tìm nhị phân (dạng "truy vấn mức `t`")

Nếu cần trả lời **nhiều truy vấn** "bỏ bao nhiêu nếu chọn mức `t`?", sắp một lần, tính tổng tiền tố; mỗi truy vấn: `k = bisect_left(a, t)` túi nhỏ hơn bị đổ (`prefix[k]`), `n − k` túi còn lại bỏ `(suffix sum) − t·(n − k)`. Đáp án bài gốc = min trên các `t` có trong mảng.

In [11]:
def removal_prefix_query(beans):
    a = sorted(beans); n = len(a)
    pre = [0] * (n + 1)
    for i, v in enumerate(a):
        pre[i + 1] = pre[i] + v
    def cost(t):
        k = bisect_left(a, t)
        return pre[k] + (pre[n] - pre[k]) - t * (n - k)
    return min(cost(t) for t in set(a)), cost

for beans_, exp in examples:
    best, cost = removal_prefix_query(beans_)
    print(best, "(kỳ vọng", exp, ")", "| truy vấn t=3:", cost(3), "| t=5:", cost(5))

4 (kỳ vọng 4 ) | truy vấn t=3: 7 | t=5: 6
7 (kỳ vọng 7 ) | truy vấn t=3: 11 | t=5: 12


**Độ phức tạp:** tiền xử lý `O(n log n)`, mỗi truy vấn `O(log n)`.

## 8. TRACE từng bước

### 8.1 Trace cách 3 trên hai ví dụ

In [12]:
def trace_sort(beans):
    a = sorted(beans); n = len(a); S = sum(a)
    print(f"  sắp: {a}, tổng = {S}")
    best = -1
    for i, v in enumerate(a):
        keep = v * (n - i)
        mark = ""
        if keep > best:
            best = keep; mark = "  <- tốt nhất mới"
        print(f"    i={i}: t = {v}, giữ {v} × {n - i} = {keep}{mark}")
    print(f"  => bỏ = {S} - {best} = {S - best}")
    return S - best

for beans_, _ in examples:
    trace_sort(beans_)

  sắp: [1, 4, 5, 6], tổng = 16
    i=0: t = 1, giữ 1 × 4 = 4  <- tốt nhất mới
    i=1: t = 4, giữ 4 × 3 = 12  <- tốt nhất mới
    i=2: t = 5, giữ 5 × 2 = 10
    i=3: t = 6, giữ 6 × 1 = 6
  => bỏ = 16 - 12 = 4
  sắp: [2, 2, 3, 10], tổng = 17
    i=0: t = 2, giữ 2 × 4 = 8  <- tốt nhất mới
    i=1: t = 2, giữ 2 × 3 = 6
    i=2: t = 3, giữ 3 × 2 = 6
    i=3: t = 10, giữ 10 × 1 = 10  <- tốt nhất mới
  => bỏ = 17 - 10 = 7


### 8.2 Trace cách 4 (đếm theo giá trị) trên mảng tự dựng có giá trị lặp (đã đối chiếu vét cạn — đối chiếu lại đề gốc)

In [13]:
def trace_counting(beans):
    A = max(beans); cnt = [0] * (A + 1)
    for b in beans: cnt[b] += 1
    ge = best = 0
    for v in range(A, 0, -1):
        ge += cnt[v]
        keep = v * ge
        if cnt[v] or keep > best:
            print(f"    v={v}: cnt={cnt[v]}, số túi ≥ v = {ge}, giữ {keep}{'  <- tốt nhất' if keep > best else ''}")
        best = max(best, keep)
    print(f"  => bỏ = {sum(beans)} - {best} = {sum(beans) - best}")
    return sum(beans) - best

demo = [3, 3, 3, 8, 1, 8, 2]
r = trace_counting(demo)
assert r == removal_brute(demo)

    v=8: cnt=2, số túi ≥ v = 2, giữ 16  <- tốt nhất
    v=3: cnt=3, số túi ≥ v = 5, giữ 15
    v=2: cnt=1, số túi ≥ v = 6, giữ 12
    v=1: cnt=1, số túi ≥ v = 7, giữ 7
  => bỏ = 28 - 16 = 12


### 8.3 Trace cách 5 — trả lời truy vấn "chọn mức `t` thì bỏ bao nhiêu?"

Với mỗi `t`, `k = bisect_left(a, t)` túi nhỏ hơn bị đổ hết (chi phí `pre[k]`), `n − k` túi còn lại mỗi túi bỏ phần vượt `t`: tổng `(pre[n] − pre[k]) − t·(n − k)`.

In [14]:
def trace_queries(beans, ts):
    a = sorted(beans); n = len(a)
    pre = [0]
    for v in a:
        pre.append(pre[-1] + v)
    print(f"  a = {a}, pre = {pre}")
    for t in ts:
        k = bisect_left(a, t)
        dump = pre[k]; trim = (pre[n] - pre[k]) - t * (n - k)
        print(f"    t={t:>2}: k={k} túi < t -> đổ {dump}; {n - k} túi ≥ t -> bớt {trim}; tổng bỏ {dump + trim} (kiểm: {removal_for(beans, t)})")

trace_queries([2, 10, 3, 2], [1, 2, 3, 4, 10, 11])

  a = [2, 2, 3, 10], pre = [0, 2, 4, 7, 17]
    t= 1: k=0 túi < t -> đổ 0; 4 túi ≥ t -> bớt 13; tổng bỏ 13 (kiểm: 13)
    t= 2: k=0 túi < t -> đổ 0; 4 túi ≥ t -> bớt 9; tổng bỏ 9 (kiểm: 9)
    t= 3: k=2 túi < t -> đổ 4; 2 túi ≥ t -> bớt 7; tổng bỏ 11 (kiểm: 11)
    t= 4: k=3 túi < t -> đổ 7; 1 túi ≥ t -> bớt 6; tổng bỏ 13 (kiểm: 13)
    t=10: k=3 túi < t -> đổ 7; 1 túi ≥ t -> bớt 0; tổng bỏ 7 (kiểm: 7)
    t=11: k=4 túi < t -> đổ 17; 0 túi ≥ t -> bớt 0; tổng bỏ 17 (kiểm: 17)


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minimumRemoval(self, beans: List[int]) -> int:
        beans.sort()
        n = len(beans)
        best_keep = 0
        for i, v in enumerate(beans):
            keep = v * (n - i)                # chọn mức t = v: giữ v hạt ở n - i túi
            if keep > best_keep:
                best_keep = keep
        return sum(beans) - best_keep

for beans_, exp in examples:
    assert Solution().minimumRemoval(list(beans_)) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Mô hình.** Mọi trạng thái cuối hợp lệ có một mức chung `t` cho các túi không rỗng (hoặc mọi túi rỗng). Với `t` cố định, cách bỏ **ít nhất** là: túi `< t` đổ hết (không còn lựa chọn), túi `≥ t` giữ đúng `t` (giữ ít hơn thì phải đổ hết — tệ hơn giữ `t`… trừ khi đổ hết, nhưng giữ `t` luôn tốt hơn đổ hết vì `t ≥ 1`). ⇒ giữ `t · #{b ≥ t}`.
2. **Chỉ xét giá trị trong mảng** (bổ đề mục 6).
3. **Công thức sắp xếp.** Với mảng tăng `a`, và giá trị `v`, gọi `i₀` là vị trí đầu tiên của `v`: `#{b ≥ v} = n − i₀`. Các vị trí `i > i₀` cùng giá trị cho `v · (n − i) < v · (n − i₀)` — không vượt quá giá trị thật ⇒ `max_i a[i]·(n−i)` = `max_v v·#{b ≥ v}`.
4. **Cách 4** tính đúng `ge = #{b ≥ v}` nhờ cộng dồn từ trên xuống.

### 10.1 Kiểm chứng bổ đề bằng thực nghiệm

Với nhiều mảng ngẫu nhiên, so sánh `max keep` trên **mọi** `t ∈ [1, A]` với `max keep` chỉ trên các `t` có trong mảng; và kiểm rằng với mọi `t` không có trong mảng, giá trị `t'` (nhỏ nhất trong mảng mà `≥ t`) cho kết quả **lớn hơn hẳn**.

In [16]:
rnd_l = random.Random(10)
strict = 0
for _ in range(2000):
    b = [rnd_l.randint(1, 30) for _ in range(rnd_l.randint(1, 8))]
    vals = sorted(set(b))
    full = max(t * sum(x >= t for x in b) for t in range(1, max(b) + 1))
    only = max(t * sum(x >= t for x in b) for t in vals)
    assert full == only
    for t in range(1, max(b) + 1):
        if t not in vals:
            t2 = min(v for v in vals if v >= t)
            assert t2 * sum(x >= t2 for x in b) > t * sum(x >= t for x in b)
            strict += 1
print("Bổ đề đúng trên 2000 mảng; số lần kiểm 't ngoài mảng kém hơn hẳn t'':", strict)

Bổ đề đúng trên 2000 mảng; số lần kiểm 't ngoài mảng kém hơn hẳn t'': 38837


## 11. Phân tích Time / Space Complexity

Gọi `n = len(beans)`, `A = max(beans)`, `V` = số giá trị khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi `t ∈ [1, A]` | `O(n · A)` | `O(1)` |
| 2. Mọi giá trị có trong mảng | `O(n · V)` | `O(V)` |
| 3. Sắp + công thức | `O(n log n)` | `O(1)`–`O(n)` |
| 4. Đếm theo giá trị | `O(n + A)` | `O(A)` |
| 5. Tiền tố + bisect | `O(n log n)` + `O(log n)`/truy vấn | `O(n)` |

In [17]:
rnd = random.Random(2171)
big = [rnd.randint(1, 10**5) for _ in range(10**5)]
for name, fn in [("sort", removal_sort), ("counting", removal_counting),
                 ("prefix_query", lambda b: removal_prefix_query(b)[0]), ("Solution", lambda b: Solution().minimumRemoval(list(b)))]:
    t0 = time.perf_counter(); r = fn(big); dt = time.perf_counter() - t0
    print(f"{name:<13}: {r}  {dt * 1000:7.1f} ms")
small = big[:300]
t0 = time.perf_counter(); removal_values(small); print(f"values (n=300): {(time.perf_counter() - t0) * 1000:.1f} ms")

sort         : 2497607025     23.9 ms
counting     : 2497607025      9.5 ms


prefix_query : 2497607025     48.2 ms
Solution     : 2497607025     27.6 ms
values (n=300): 3.1 ms


## 12. Bộ test

In [18]:
ALL = {
    "values": removal_values,
    "sort": removal_sort,
    "counting": removal_counting,
    "prefix_query": lambda b: removal_prefix_query(b)[0],
    "Solution": lambda b: Solution().minimumRemoval(list(b)),
}

def check(beans, exp=None, brute=True, skip=()):
    ref = removal_brute(beans) if brute else removal_sort(beans)
    for name, fn in ALL.items():
        if name in skip:
            continue
        got = fn(list(beans))
        assert got == ref, (name, beans, got, ref)
    if exp is not None:
        assert ref == exp, (beans, ref, exp)
    return ref

for beans_, exp in examples:
    check(beans_, exp)
print("Ví dụ đề: OK")

check([5], 0)                        # một túi
check([7, 7, 7], 0)                  # đã bằng nhau
check([1, 100], 1)                   # đổ túi nhỏ
check([50, 51], 1)                   # bớt túi lớn
check([1, 1, 1, 1, 10], 4)           # giữ 1 ở 5 túi: bỏ 9; giữ 10 ở 1 túi: bỏ 4
check([1, 2, 3, 4, 5], 6)            # t = 3: giữ 9, bỏ 6
check([10**5] * 10**5, 0, brute=False, skip=("values",))
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(3000):
    n = rnd.randint(1, 10)
    check([rnd.randint(1, 12) for _ in range(n)])
print("3000 mảng nhỏ so với vét cạn: OK")

for _ in range(10):
    n = rnd.randint(1, 10**5)
    check([rnd.randint(1, rnd.choice([10, 1000, 10**5])) for _ in range(n)], brute=False, skip=("values",))
print("10 mảng lớn (đối chiếu chéo): OK")

Ví dụ đề: OK


Edge case: OK
3000 mảng nhỏ so với vét cạn: OK


10 mảng lớn (đối chiếu chéo): OK


In [19]:
# Kiểm các tính chất (property-based) của đáp án
rnd_p = random.Random(12)
for _ in range(2000):
    b = [rnd_p.randint(1, 50) for _ in range(rnd_p.randint(1, 12))]
    ans = Solution().minimumRemoval(list(b))
    S = sum(b); M = max(b)
    assert 0 <= ans <= S - M                           # luôn giữ được ít nhất túi lớn nhất
    assert ans == Solution().minimumRemoval(list(reversed(b)))      # không phụ thuộc thứ tự
    k = rnd_p.randint(1, 5)
    assert Solution().minimumRemoval([x * k for x in b]) == k * ans  # nhân đều số hạt -> nhân đều đáp án
    assert Solution().minimumRemoval(b + [M]) <= ans + M             # thêm túi = max không làm tệ hơn quá M
print("Tính chất: 0 ≤ đáp án ≤ tổng − max, bất biến thứ tự, thuần nhất khi nhân k — đúng trên 2000 mảng")

Tính chất: 0 ≤ đáp án ≤ tổng − max, bất biến thứ tự, thuần nhất khi nhân k — đúng trên 2000 mảng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chọn mức `t` = giá trị **nhỏ nhất** (không đổ túi nào)

In [20]:
def bug_min_level(beans):
    t = min(beans)
    return sum(b - t for b in beans)

print("[2,10,3,2] -> sai:", bug_min_level([2, 10, 3, 2]), "| đúng:", Solution().minimumRemoval([2, 10, 3, 2]))

[2,10,3,2] -> sai: 9 | đúng: 7


**Vì sao sai:** quên rằng ta được **đổ hết** một túi; với túi 10 hạt, giữ nguyên và đổ các túi nhỏ rẻ hơn. **Sửa:** thử mọi mức có trong mảng.

### Lỗi 2 — Chọn `t` = trung bình hoặc trung vị

In [21]:
def bug_median(beans):
    a = sorted(beans); t = a[len(a) // 2]
    return sum(b if b < t else b - t for b in a)

def bug_mean(beans):
    t = round(sum(beans) / len(beans))
    return sum(b if b < t else b - t for b in beans)

for b in [[4, 1, 6, 5], [1, 1, 1, 1, 10]]:
    print(f"{b}: trung vị -> {bug_median(b)}, trung bình -> {bug_mean(b)}, đúng -> {Solution().minimumRemoval(list(b))}")

[4, 1, 6, 5]: trung vị -> 6, trung bình -> 4, đúng -> 4
[1, 1, 1, 1, 10]: trung vị -> 9, trung bình -> 11, đúng -> 4


**Vì sao sai:** hàm mục tiêu `t · #{b ≥ t}` **không** tối ưu ở trung vị/trung bình (đó là bài "tổng khoảng cách nhỏ nhất", khác bài này). **Sửa:** duyệt mọi giá trị.

### Lỗi 3 — Nhầm số túi giữ lại: dùng `i + 1` (túi ≤ t) thay vì `n − i`

In [22]:
def bug_count_side(beans):
    a = sorted(beans); n = len(a)
    return sum(a) - max(a[i] * (i + 1) for i in range(n))       # SAI: đếm túi <= t

print("[4,1,6,5] -> sai:", bug_count_side([4, 1, 6, 5]), "| đúng:", Solution().minimumRemoval([4, 1, 6, 5]))

[4,1,6,5] -> sai: -8 | đúng: 4


**Vì sao sai:** túi **nhỏ hơn** `t` bị đổ, túi **lớn hơn hoặc bằng** được giữ. Kết quả âm (`16 − 24 = −8`) là dấu hiệu rõ ràng sai. **Sửa:** `n − i`.

### Lỗi 4 — Chỉ bớt các túi lớn, quên đổ túi nhỏ (tính sai chi phí)

In [23]:
def bug_forget_small(beans):
    best = None
    for t in set(beans):
        c = sum(b - t for b in beans if b > t)                 # SAI: túi < t cũng phải đổ
        best = c if best is None else min(best, c)
    return best

print("[2,10,3,2] -> sai:", bug_forget_small([2, 10, 3, 2]), "| đúng:", Solution().minimumRemoval([2, 10, 3, 2]))

[2,10,3,2] -> sai: 0 | đúng: 7


**Vì sao sai:** chọn `t = 10` "miễn phí" vì quên chi phí đổ các túi 2, 3, 2. **Sửa:** cộng cả `b` với mọi `b < t`.

### Lỗi 5 — Sắp **giảm dần** nhưng giữ công thức `a[i] × (n − i)`

In [24]:
def bug_desc(beans):
    a = sorted(beans, reverse=True); n = len(a)
    return sum(a) - max(a[i] * (n - i) for i in range(n))      # SAI: với mảng giảm phải là (i + 1)

print("[4,1,6,5] -> sai:", bug_desc([4, 1, 6, 5]), "| đúng:", Solution().minimumRemoval([4, 1, 6, 5]))

[4,1,6,5] -> sai: -8 | đúng: 4


**Vì sao sai:** trong mảng **giảm**, số túi `≥ a[i]` là `i + 1`. Công thức và chiều sắp phải đi cùng nhau. Kết quả âm cũng là dấu hiệu sai.

### Lỗi 6 — Tìm tam phân (ternary search) trên `t` vì "trông giống hàm lồi"

In [25]:
def bug_ternary(beans):
    S = sum(beans)
    f = lambda t: t * sum(b >= t for b in beans)
    lo, hi = 1, max(beans)
    while hi - lo > 2:
        m1 = lo + (hi - lo) // 3; m2 = hi - (hi - lo) // 3
        if f(m1) < f(m2):
            lo = m1
        else:
            hi = m2
    return S - max(f(t) for t in range(lo, hi + 1))

rnd_t = random.Random(6)
for _ in range(5000):
    b = [rnd_t.randint(1, 20) for _ in range(rnd_t.randint(2, 7))]
    if bug_ternary(b) != Solution().minimumRemoval(list(b)):
        break
print(f"{b}: tam phân (sai) -> {bug_ternary(b)}, đúng -> {Solution().minimumRemoval(list(b))}")
print("   đường cong giữ lại:", [k for _, k in keep_curve(b)])

[16, 8, 5, 20, 7]: tam phân (sai) -> 28, đúng -> 24
   đường cong giữ lại: [5, 10, 15, 20, 25, 24, 28, 24, 18, 20, 22, 24, 26, 28, 30, 32, 17, 18, 19, 20]


**Vì sao sai:** tìm tam phân chỉ đúng với hàm **một đỉnh** (unimodal). `keep(t)` nhảy bậc mỗi khi qua một giá trị của mảng nên có nhiều đỉnh cục bộ (đường cong in ở trên). Phản ví dụ được tìm tự động bằng cách so với lời giải đúng. **Sửa:** duyệt mọi giá trị (sau khi sắp, chỉ `O(n)`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Làm các phần tử bằng nhau bằng cách **chỉ giảm** (hoặc xóa)" ⇒ thử mức `t` thuộc mảng, sắp xếp để đếm số phần tử `≥ t`.
- "Bỏ ít nhất" = "tổng − giữ nhiều nhất".
- Mô hình `t × số phần tử ≥ t` giống bài "H-index" và "Maximum Units"/"Maximum Ice Cream".
- Đáp án âm hoặc lớn hơn tổng ⇒ chắc chắn công thức sai — tự kiểm nhanh.
- Kiểu dữ liệu: tổng tới `10¹⁰` — trong C++/Java dùng `long long`/`long`.
- Hàm mục tiêu có dạng "bậc thang × tuyến tính" ⇒ **không** lồi ⇒ không tìm nhị phân/tam phân, hãy liệt kê điểm gãy (các giá trị của mảng).
- Khi cần cả **nhiều truy vấn** mức `t`: sắp + tổng tiền tố + `bisect` (cách 5).
- Để chắc chắn trước khi nộp: in bảng `t → giữ` cho ví dụ nhỏ như mục 4 — dễ phát hiện sai công thức đếm.

## 15. Biến thể của bài

1. **LeetCode 274 — H-Index:** max `h` sao cho có `≥ h` bài có `≥ h` trích dẫn — cùng cấu trúc "giá trị × số phần tử ≥".
2. **LeetCode 1648 — Sell Diminishing-Valued Colored Balls.**
3. **Không được đổ hết túi** (mọi túi phải còn ≥ 1 hạt) ⇒ `t = min(beans)` bắt buộc.
4. **Được phép cả thêm hạt** với chi phí khác — bài "Minimum Cost to Make Array Equal" (2448), hàm lồi, tìm tam phân/trung vị có trọng số.
5. **Tối đa `k` túi được giữ** (bài tập 5).

In [26]:
def h_index(citations):
    """LeetCode 274 bằng cùng ý tưởng: sắp tăng, h = max min(a[i], n - i)."""
    a = sorted(citations); n = len(a)
    return max([min(a[i], n - i) for i in range(n)] + [0])

def h_index_brute(c):
    return max(h for h in range(len(c) + 1) if sum(x >= h for x in c) >= h)

rnd_v = random.Random(15)
for _ in range(1000):
    c = [rnd_v.randint(0, 10) for _ in range(rnd_v.randint(1, 10))]
    assert h_index(c) == h_index_brute(c)
print("H-index [3,0,6,1,5] ->", h_index([3, 0, 6, 1, 5]), "| khớp vét cạn trên 1000 mảng")

H-index [3,0,6,1,5] -> 3 | khớp vét cạn trên 1000 mảng


### 15.1 So sánh với biến thể 4 (được thêm **và** bớt, chi phí theo đơn vị)

LeetCode 2448: mỗi phần tử `nums[i]` đổi một đơn vị tốn `cost[i]`, làm mọi phần tử bằng nhau. Hàm chi phí `Σ cost[i]·|nums[i] − t|` là **lồi** ⇒ tối ưu tại **trung vị có trọng số**. Khác hẳn bài 2171 (không lồi, vì "đổ hết túi" tạo bước nhảy). Cell dưới cài đặt trung vị có trọng số và kiểm vét cạn — để thấy khi nào trung vị là đúng.

In [27]:
def min_cost_equal(nums, cost):
    pairs = sorted(zip(nums, cost)); total = sum(cost); acc = 0
    for v, c in pairs:
        acc += c
        if 2 * acc >= total:
            t = v; break
    return sum(c * abs(v - t) for v, c in zip(nums, cost))

rnd_w = random.Random(15)
for _ in range(1000):
    n = rnd_w.randint(1, 7)
    nums = [rnd_w.randint(1, 20) for _ in range(n)]; cost = [rnd_w.randint(1, 9) for _ in range(n)]
    assert min_cost_equal(nums, cost) == min(sum(c * abs(v - t) for v, c in zip(nums, cost)) for t in range(1, 21))
print("2448 [1,3,5,2] cost [2,3,1,14] ->", min_cost_equal([1, 3, 5, 2], [2, 3, 1, 14]), "| trung vị có trọng số khớp vét cạn")

2448 [1,3,5,2] cost [2,3,1,14] -> 8 | trung vị có trọng số khớp vét cạn


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi `t` | thử `1..A` | `O(nA)` | `O(1)` | kiểm thử |
| 2. Mọi giá trị | bổ đề "t trong mảng" | `O(nV)` | `O(V)` | bước trung gian |
| 3. Sắp + công thức | `a[i]·(n−i)` | `O(n log n)` | `O(1)` | **nộp bài** |
| 4. Đếm theo giá trị | `v · ge[v]` | `O(n + A)` | `O(A)` | `A` nhỏ |
| 5. Tiền tố + bisect | truy vấn `O(log n)` | `O(n log n)` | `O(n)` | nhiều truy vấn `t` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimumRemoval([1, 2, 3, 4, 5])`?

**Bài 2 (tính tay):** Với `[3, 3, 3, 8, 1, 8, 2]`, lập bảng `t`, số túi `≥ t`, giữ, bỏ cho mọi `t` có trong mảng.

**Bài 3 (điền TODO):** Hoàn thiện `best_level(beans)` — trả về **cặp** `(số hạt bỏ, mức t)`; khi hòa chọn `t` nhỏ nhất.

**Bài 4 (tự viết lại):** Viết lại cách 3 trả về luôn **mảng kết quả** sau khi bỏ hạt (cùng thứ tự với đầu vào) và kiểm bằng validator: mọi túi khác 0 bằng nhau, không túi nào tăng, tổng bỏ = đáp án.

**Bài 5 (mini challenge):** Thêm ràng buộc: được giữ **tối đa `k`** túi không rỗng. Tìm số hạt bỏ ít nhất. Gợi ý: với `t = a[i]` (sắp tăng), số túi giữ là `min(k, n − i)`.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_level(beans):
    # TODO: duyệt mảng sắp tăng, giữ (keep lớn nhất, t nhỏ nhất khi hòa)
    pass

# Bài 4 — TODO
def remove_and_show(beans):
    # TODO: tìm t tối ưu, rồi [0 if b < t else t for b in beans]
    pass

# Bài 5 — TODO
def removal_at_most_k(beans, k):
    # TODO: max a[i] * min(k, n - i)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Tổng 15.
- Mức 2: Giữ: `1·5, 2·4, 3·3, 4·2, 5·1` = `5, 8, 9, 8, 5`.
- Mức 3: `15 − 9 = 6`.

**Bài 2**
- Mức 1: Sắp: `[1, 2, 3, 3, 3, 8, 8]`, tổng 28.
- Mức 2: `t = 3`: 5 túi ⇒ giữ 15; `t = 8`: 2 túi ⇒ giữ 16.
- Mức 3: Tối ưu `t = 8`, bỏ `12`.

**Bài 3**
- Mức 1: So sánh `keep > best` (nghiêm ngặt) khi duyệt `t` tăng ⇒ giữ `t` nhỏ nhất khi hòa.
- Mức 2: Trả `(tổng − best, t)`.
- Mức 3: Kiểm với vét cạn mọi `t`.

**Bài 4**
- Mức 1: Dùng `t` từ bài 3.
- Mức 2: Túi `< t` → 0, túi `≥ t` → `t`.
- Mức 3: Validator: `set(x for x in out if x) ≤ {t}`, `out[i] ≤ beans[i]`.

**Bài 5**
- Mức 1: Giữ tối đa `k` túi trong số `n − i` túi `≥ t`.
- Mức 2: Giữ `t · min(k, n − i)`.
- Mức 3: Đối chiếu vét cạn mọi `t` và mọi tập `k` túi (n nhỏ).

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
r1 = Solution().minimumRemoval([1, 2, 3, 4, 5])
print("Bài 1:", r1); assert r1 == 6

# Bài 2
b2 = [3, 3, 3, 8, 1, 8, 2]; S = sum(b2)
for t in sorted(set(b2)):
    k = sum(x >= t for x in b2)
    print(f"  Bài 2: t={t}: {k} túi, giữ {t * k}, bỏ {S - t * k}")
assert Solution().minimumRemoval(list(b2)) == 12

# Bài 3
def best_level_sol(beans):
    a = sorted(beans); n = len(a); best, bt = -1, None
    for i, v in enumerate(a):
        if v * (n - i) > best:
            best, bt = v * (n - i), v
    return sum(a) - best, bt

rnd = random.Random(3)
for _ in range(1500):
    b = [rnd.randint(1, 10) for _ in range(rnd.randint(1, 10))]
    c, t = best_level_sol(b)
    opts = [(sum(b) - u * sum(x >= u for x in b), u) for u in range(1, max(b) + 1)]
    assert (c, t) == min(opts)
print("Bài 3 OK, ví dụ:", best_level_sol([4, 1, 6, 5]), best_level_sol([2, 10, 3, 2]))

Bài 1: 6
  Bài 2: t=1: 7 túi, giữ 7, bỏ 21
  Bài 2: t=2: 6 túi, giữ 12, bỏ 16
  Bài 2: t=3: 5 túi, giữ 15, bỏ 13
  Bài 2: t=8: 2 túi, giữ 16, bỏ 12
Bài 3 OK, ví dụ: (4, 4) (7, 10)


In [30]:
# Bài 4
def remove_and_show_sol(beans):
    _, t = best_level_sol(beans)
    return [0 if b < t else t for b in beans]

def valid_result(beans, out):
    nz = {x for x in out if x}
    return len(nz) <= 1 and all(0 <= o <= b for o, b in zip(out, beans)) and sum(beans) - sum(out) == Solution().minimumRemoval(list(beans))

# Bài 5
def removal_at_most_k_sol(beans, k):
    a = sorted(beans); n = len(a)
    return sum(a) - max(a[i] * min(k, n - i) for i in range(n))

from itertools import combinations
def at_most_k_brute(beans, k):
    best = sum(beans)
    for t in range(1, max(beans) + 1):
        idx = [i for i, b in enumerate(beans) if b >= t]
        for r in range(1, min(k, len(idx)) + 1):
            for comb in combinations(idx, r):
                best = min(best, sum(beans) - t * r)
    return best

rnd = random.Random(5)
for _ in range(800):
    b = [rnd.randint(1, 8) for _ in range(rnd.randint(1, 7))]
    assert valid_result(b, remove_and_show_sol(b))
    k = rnd.randint(1, len(b))
    assert removal_at_most_k_sol(b, k) == at_most_k_brute(b, k)
print("Bài 4, Bài 5 OK; ví dụ:", remove_and_show_sol([4, 1, 6, 5]), removal_at_most_k_sol([4, 1, 6, 5], 2))

Bài 4, Bài 5 OK; ví dụ: [4, 0, 4, 4] 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
a = sorted(beans); n = len(a)
return sum(a) - max(a[i] * (n - i) for i in range(n))
```

**Checklist:**
- [ ] Túi `< t` **đổ hết**, túi `≥ t` bớt về `t`?
- [ ] Mảng tăng ⇒ `n − i`; mảng giảm ⇒ `i + 1`?
- [ ] Thử **mọi** giá trị có trong mảng (không dùng trung vị/trung bình)?
- [ ] Đáp án nằm trong `[0, tổng]` (tự kiểm)?
- [ ] C++/Java: dùng kiểu 64-bit?
- [ ] Chữ ký đúng: `def minimumRemoval(self, beans: List[int]) -> int`?